# Day 054 — Exercise 3: Reading the Data Back

**What you'll build:** `get_messages(session, conversation_id)` returning ordered `[{'role','content'}]` dicts, and `list_conversations(session)` returning `[{'id','title','message_count'}]`.

**Why it matters:** Saving is half the story — the app has to read state back to rebuild a chat. `get_messages` returns plain dicts (safe to use after the session closes, and ready to feed straight to the model). `list_conversations` powers a sidebar of past chats.

## Provided: Setup + Models + create/add helpers

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
import tempfile
from datetime import datetime
from sqlalchemy import create_engine, ForeignKey, select, inspect as sa_inspect, text
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Conversation(Base):
    """One chat conversation. Has many Messages (one-to-many)."""
    __tablename__ = 'conversations'

    id:         Mapped[int]      = mapped_column(primary_key=True)
    title:      Mapped[str]      = mapped_column(default='New chat')
    created_at: Mapped[datetime] = mapped_column(default=datetime.utcnow)

    # relationship() is the ORM link (not a DB column). cascade deletes a
    # conversation's messages when the conversation is deleted.
    messages: Mapped[list['Message']] = relationship(
        back_populates='conversation', cascade='all, delete-orphan')


class Message(Base):
    """One message in a conversation. Belongs to one Conversation (many-to-one)."""
    __tablename__ = 'messages'

    id:              Mapped[int]      = mapped_column(primary_key=True)
    conversation_id: Mapped[int]      = mapped_column(ForeignKey('conversations.id'))
    role:            Mapped[str]      = mapped_column()
    content:         Mapped[str]      = mapped_column()
    created_at:      Mapped[datetime] = mapped_column(default=datetime.utcnow)

    conversation: Mapped['Conversation'] = relationship(back_populates='messages')


def memory_engine():
    """In-memory SQLite engine for tests. StaticPool makes every Session share the
    one in-memory database (see Day 44)."""
    return create_engine('sqlite:///:memory:',
                          connect_args={'check_same_thread': False},
                          poolclass=StaticPool)


def create_schema(engine) -> None:
    """Create every table registered on Base (CREATE TABLE IF NOT EXISTS)."""
    Base.metadata.create_all(engine)


def create_conversation(session, title: str = 'New chat') -> Conversation:
    """Insert a new conversation and flush so its auto id is assigned.
    The caller controls commit (unit-of-work pattern)."""
    conv = Conversation(title=title)
    session.add(conv)
    session.flush()
    return conv


def add_message(session, conversation_id: int, role: str, content: str) -> Message:
    """Append a message to a conversation via its foreign key, and flush to assign
    the id. The caller commits."""
    msg = Message(conversation_id=conversation_id, role=role, content=content)
    session.add(msg)
    session.flush()
    return msg

## Your Implementation

In [ ]:
def get_messages(session, conversation_id: int) -> list:
    """Return the conversation's messages in insertion order as
    [{'role', 'content'}] dicts."""
    # TODO: stmt = (select(Message)
    #                .where(Message.conversation_id == conversation_id)
    #                .order_by(Message.id))
    # TODO: rows = session.execute(stmt).scalars().all()
    # TODO: return [{'role': m.role, 'content': m.content} for m in rows]
    pass


def list_conversations(session) -> list:
    """Return [{'id', 'title', 'message_count'}] for all conversations, by id."""
    # TODO: convs = session.execute(select(Conversation).order_by(Conversation.id)).scalars().all()
    # TODO: return [{'id': c.id, 'title': c.title, 'message_count': len(c.messages)} for c in convs]
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    engine = memory_engine()
    create_schema(engine)
    with Session(engine) as s:
        conv = create_conversation(s, 'demo')
        add_message(s, conv.id, 'user', 'first')
        add_message(s, conv.id, 'assistant', 'second')
        add_message(s, conv.id, 'user', 'third')
        s.commit()
        cid = conv.id

    # Check 1: get_messages returns a list of role/content dicts
    try:
        with Session(engine) as s:
            msgs = get_messages(s, cid)
        assert isinstance(msgs, list) and all(set(m) == {'role', 'content'} for m in msgs), \
            f'expected role/content dicts, got {msgs}'
        passed += 1; print('✅ Check 1: get_messages -> [{role, content}]')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: messages come back in insertion order
    try:
        with Session(engine) as s:
            msgs = get_messages(s, cid)
        assert [m['content'] for m in msgs] == ['first', 'second', 'third'], f'order wrong: {msgs}'
        passed += 1; print('✅ Check 2: messages ordered by insertion')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: unknown conversation -> empty list (no crash)
    try:
        with Session(engine) as s:
            assert get_messages(s, 9999) == [], 'unknown conversation should give []'
        passed += 1; print('✅ Check 3: unknown conversation -> []')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: list_conversations returns id/title/message_count
    try:
        with Session(engine) as s:
            convs = list_conversations(s)
        assert convs and set(convs[0]) == {'id', 'title', 'message_count'}, f'bad shape: {convs}'
        passed += 1; print('✅ Check 4: list_conversations -> [{id, title, message_count}]')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: message_count is correct
    try:
        with Session(engine) as s:
            convs = list_conversations(s)
        row = [c for c in convs if c['id'] == cid][0]
        assert row['message_count'] == 3, f"expected 3, got {row['message_count']}"
        passed += 1; print('✅ Check 5: message_count matches')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def get_messages(session, conversation_id: int) -> list:
    """Return the conversation's messages, in insertion order, as plain
    [{'role', 'content'}] dicts (safe to use after the session closes)."""
    stmt = (select(Message)
            .where(Message.conversation_id == conversation_id)
            .order_by(Message.id))
    rows = session.execute(stmt).scalars().all()
    return [{'role': m.role, 'content': m.content} for m in rows]


def list_conversations(session) -> list:
    """Return all conversations as [{'id', 'title', 'message_count'}] by id."""
    convs = session.execute(
        select(Conversation).order_by(Conversation.id)).scalars().all()
    return [{'id': c.id, 'title': c.title, 'message_count': len(c.messages)}
            for c in convs]
```

**Why this works:** `select(Message).where(...).order_by(Message.id)` is the SQLAlchemy 2.x query from Day 44, now filtered by the foreign key and ordered. Converting rows to plain dicts *inside* the session means the result is detached data the caller can use anywhere — no lazy-load errors after the session closes. `list_conversations` reads each conversation's `messages` relationship for the count, giving the sidebar everything it needs in one call.
</details>